# Clase 046 — MongoDB / pymongo

**Parte 0** · MongoDB + pymongo docs.

> 🎯 Modelo documento, CRUD, queries con operadores, aggregation pipeline.

> ⏱️ ~75 min

## ⚙️ Setup

Para este notebook usamos `mongomock` para no requerir Mongo real:

```bash
pip install mongomock pymongo
```

La API es idéntica a pymongo — solo cambia `MongoClient` por `mongomock.MongoClient`.

In [ ]:
try:
    import mongomock
    client = mongomock.MongoClient()
    print('mongomock OK')
except ImportError:
    print('Instala: pip install mongomock')

db = client['lab044']
productos = db['productos']

## 1️⃣ SQL vs NoSQL — el modelo

| | SQL | NoSQL documento (Mongo) |
|---|---|---|
| Unidad | Fila en tabla | Documento JSON en collection |
| Schema | Rígido (CREATE TABLE) | Flexible (cada doc puede tener cols distintas) |
| Relaciones | JOINs entre tablas | Documentos anidados (denormalización) |
| Escala | Vertical (más CPU/RAM) | Horizontal (más nodos) |
| Transacciones ACID | Sí, fuerte | Sí pero más limitado |
| Mejor para | Datos estructurados, integridad referencial | Schema variable, datos jerárquicos, alta escala write |

## 2️⃣ Insert

In [ ]:
docs = [
    {'nombre': 'Libro Python', 'categoria': 'libros',  'precio': 30, 'stock': 50, 'tags': ['programacion', 'python']},
    {'nombre': 'Guitarra',     'categoria': 'musica',  'precio': 800, 'stock': 5,  'tags': ['acustica']},
    {'nombre': 'Auriculares',  'categoria': 'audio',   'precio': 120, 'stock': 30, 'tags': ['wireless', 'bluetooth']},
    {'nombre': 'Libro Pandas', 'categoria': 'libros',  'precio': 35, 'stock': 40, 'tags': ['programacion', 'datos']},
    {'nombre': 'Mouse',        'categoria': 'audio',   'precio': 45, 'stock': 100, 'tags': ['wireless']},
]
result = productos.insert_many(docs)
print(f'insertados: {len(result.inserted_ids)} docs')

## 3️⃣ Find con operadores

| Operador | Equivalente SQL |
|---|---|
| `$gt`, `$lt`, `$gte`, `$lte` | `>`, `<`, `>=`, `<=` |
| `$eq`, `$ne` | `=`, `<>` |
| `$in`, `$nin` | `IN`, `NOT IN` |
| `$and`, `$or`, `$not` | `AND`, `OR`, `NOT` |
| `$regex` | `LIKE` con regex |
| `$exists` | `IS NULL`/`IS NOT NULL` |

In [ ]:
# Productos > 100 y en categorías específicas
query = {
    'precio': {'$gt': 100},
    'categoria': {'$in': ['libros', 'audio']},
}
for doc in productos.find(query):
    print(f"{doc['nombre']:20s}  cat={doc['categoria']:8s}  €{doc['precio']}")

## 4️⃣ Update con `$set` y `$inc`

In [ ]:
productos.update_one(
    {'nombre': 'Auriculares'},
    {'$inc': {'stock': 10}, '$set': {'oferta': True}}
)
print(productos.find_one({'nombre': 'Auriculares'}))

## 5️⃣ Aggregation pipeline

Pipeline de etapas — análogo a SQL pero más componible:

In [ ]:
pipeline = [
    {'$match': {'precio': {'$lt': 500}}},
    {'$group': {
        '_id': '$categoria',
        'n': {'$sum': 1},
        'precio_medio': {'$avg': '$precio'},
    }},
    {'$sort': {'precio_medio': -1}},
]
for doc in productos.aggregate(pipeline):
    print(doc)

## 6️⃣ Documentos jerárquicos — el valor real de NoSQL

Una review embebida en un producto evita un JOIN. `$elemMatch` filtra por condiciones en sub-documentos:

In [ ]:
productos.update_one(
    {'nombre': 'Libro Python'},
    {'$set': {'reviews': [
        {'user': 'ana', 'rating': 5, 'comentario': 'excelente'},
        {'user': 'bob', 'rating': 2, 'comentario': 'no me gustó'},
        {'user': 'cris', 'rating': 4, 'comentario': 'bueno'},
    ]}}
)

# Buscar productos con alguna review baja
malos = list(productos.find({'reviews': {'$elemMatch': {'rating': {'$lt': 3}}}}))
for m in malos:
    print(f"{m['nombre']} tiene reviews bajas")

## 🚫 Cuándo NO usar Mongo

- **Necesitas integridad referencial fuerte** (relaciones ⊥ desnormalización).
- **Tu modelo es naturalmente tabular** — usar Mongo añade complejidad sin ganancia.
- **Reporting y BI son críticos** — SQL es mucho mejor para analítica.
- **Tu equipo no quiere aprender otro paradigma** — costo de oportunidad.

## ✅ Checklist

- [ ] Entiendo modelo documento vs relacional
- [ ] Hago CRUD con pymongo
- [ ] Uso operadores ($gt, $in, $regex, etc.)
- [ ] Hago aggregation pipeline ($match/$group/$sort)
- [ ] Sé cuándo NO conviene Mongo

## 📝 Homework

Ver `README.md`. 20 productos, 5 queries, pipeline, reporte cuándo Mongo vs SQL.

## 📖 Definiciones y características

**NoSQL documento**

Familia de DBs que almacena **documentos JSON-like** (BSON en Mongo) en lugar de filas en tablas. Schema flexible — cada documento puede tener campos distintos.

**Collection**

Equivalente a una tabla en SQL, pero sin schema fijo. Contiene documentos del mismo "tipo" lógico (productos, usuarios, eventos).

**Documento (`dict` BSON)**

Unidad de almacenamiento. JSON con tipos extra (Date, ObjectId, Decimal128). Puede contener arrays y sub-documentos anidados.

**Operador (`$gt`, `$in`, `$elemMatch`)**

Prefijo `$` en las queries Mongo: `{'precio': {'$gt': 100}}` ≈ `WHERE precio > 100`. La query es un dict JSON, no string.

**Aggregation pipeline**

Equivalente Mongo a CTEs encadenadas: lista de etapas (`$match`, `$group`, `$sort`, `$project`, `$lookup`) que procesan documentos secuencialmente.

**`$elemMatch`**

Operador para filtrar por condiciones sobre **elementos de un array** dentro del documento. Útil con arrays de sub-docs (reviews dentro de producto).

## ⚠️ Errores comunes

| Síntoma / mensaje | Causa y cómo arreglar |
|---|---|
| `pymongo.errors.ServerSelectionTimeoutError` | No conecta al servidor. **Fix**: verifica que Mongo está corriendo (`docker ps`), URI correcta (`mongodb://localhost:27017`), firewall. |
| Update sin `$set` reemplaza el documento entero | `update_one(filter, {'precio': 100})` reemplaza TODO el doc. **Fix**: usa `{'$set': {'precio': 100}}` para modificar solo ese campo. |
| Aggregation con `$group` sin `_id` falla | `$group` requiere `_id` (la key de agrupación, puede ser `None` para agrupar todo). **Fix**: `{'$group': {'_id': '$categoria', 'total': {'$sum': '$monto'}}}`. |
| Query con `{}` devuelve todo (no None) | `{}` es "sin filtro" en Mongo. Si querías matchear todo, OK; si querías nada, `{'_id': {'$exists': False}}` o similar. |
| Cuento con `count_documents({})` y va lento | Sin filtro, recorre toda la collection. **Fix**: para counts aproximados rápidos, `db.coll.estimated_document_count()`. |

## ❓ Preguntas frecuentes

**❓ ¿SQL o NoSQL?**

**SQL** para datos tabulares con relaciones, integridad referencial, reporting/BI. **NoSQL documento** para schema variable, datos jerárquicos naturales, escala write masiva. No es "mejor" — distinto.

**❓ ¿`find_one` o `find`?**

`find_one` devuelve dict (o None). `find` devuelve cursor iterable. Para 1 doc: `find_one`. Para muchos: `list(coll.find(query))` o iterar el cursor.

**❓ ¿pymongo vs Motor (async)?**

**pymongo** síncrono, default. **Motor** asíncrono (asyncio) — para web apps con muchas concurrent connections.

**❓ ¿Cómo tipo los documentos en Python?**

Usa **pydantic** con `BaseModel`. Convierte dict ↔ tipo validado. Combinado con FastAPI, casi gratis (verás en MLOps).

**❓ ¿Mongo para data science?**

Como **fuente** sí (extraes datos con aggregation, los pasas a pandas). Para **análisis** ya no — pandas/DuckDB son mejores. Mongo brilla en operaciones (logs, eventos).

## 🔗 Referencias

- [pymongo](https://pymongo.readthedocs.io/)
- [MongoDB operators](https://www.mongodb.com/docs/manual/reference/operator/query/)

➡️ **Siguiente:** [045 — APIs REST](../045-apis-rest-con-requests/README.md)